<a href="https://colab.research.google.com/github/isiahndanzia/CSCE1015/blob/main/Copy_of_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [1]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [2]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()
print(f"The number of missing students is {n_missing_study} students")

The number of missing students is 255 students


**Your justification (1–2 sentences):** _..._

In [3]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [4]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
clean_map = {"on-campus":"On-Campus", "off-campus":"Off-Campus",
             "off campus":"Off-Campus", "with family":"With Family"}
students["housing_clean"] = students["housing"].str.strip().str.lower().map(clean_map)

In [5]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [6]:
# TODO
impossible_ages = sorted(int(a) for a in students.loc[(students["age"] < 15) | (students["age"] > 90), "age"].unique())
n_neg_work = (students["work_hours_per_week"] < 0).sum()

In [7]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [8]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset="student_id")

**Why not de-dupe enroll / activity? (1 sentence):** _..._ If you de duplicate by enroll/activity you would remove a lot of student records because many students may have the same enrollment or activities

In [9]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [10]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = students_dedup.copy()
analysis["student_id"] = analysis["student_id"].str.replace("NU-", "").astype(int)

In [11]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [22]:
# TODO
matched = enroll["sid"].isin(set(analysis["student_id"])).sum()

In [24]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [26]:
# TODO
dates_parsed = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")

In [27]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [34]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]

analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4]
)

In [35]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

❌ C8 not yet — add a z-scored column and a 4-band gpa_band column


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [41]:
# TODO: write analysis to northgate_clean.csv
clean = students.copy()                                   # never touch the raw

# 1. de-duplicate (student_records only: one row per student)
clean = clean.drop_duplicates()                          # 18 exact copies
clean = clean.drop_duplicates(subset="student_id")       # 9 near-duplicates -> keep first

# 2. standardize categories (already computed as housing_clean)
# 3. fix impossible values -> mark missing
clean.loc[(clean["age"] < 15) | (clean["age"] > 90), "age"] = np.nan
clean.loc[clean["work_hours_per_week"] < 0, "work_hours_per_week"] = np.nan
# 4. parse dates deliberately
clean["enrollment_date"] = pd.to_datetime(clean["enrollment_date"], format="mixed", errors="coerce")
# 5. standardize the key for joining
clean["sid"] = clean["student_id"].str.replace("NU-", "", regex=False).astype(int)

print("clean student table:", clean.shape)
clean.to_csv("northgate_clean.csv", index=False)   # a NEW file; raw stays untouched
print("northgate_clean.csv written (raw files untouched)")

clean student table: (2000, 14)
northgate_clean.csv written (raw files untouched)


In [42]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- _decision → justification_
- got rid of impossible ages to make the data make sense
- parsed dates for better data
- cleaned student ids for more accurate data

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [48]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
northgate = pd.read_csv("northgate_clean.csv")
mean_gpa = northgate["final_gpa"].mean()
print(mean_gpa)

2.3053600000000003


In [50]:
northgate.head()

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out,housing_clean,sid
0,NU-101508,Data Science,27.0,Off-Campus,7.6,15.0,1,81,1.86,NaN,2023-09-13,1,Off-Campus,101508
1,NU-102438,Learning Technologies,25.0,With Family,14.3,35.0,1,100,2.06,9.5,2023-10-19,0,With Family,102438
2,NU-102385,Learning Technologies,26.0,With Family,3.4,18.0,9,77,3.41,19.1,2022-11-16,0,With Family,102385
3,NU-100767,Cyber Security,22.0,On-Campus,23.7,40.0,1,65,1.73,8.1,2024-05-03,0,On-Campus,100767
4,NU-106054,Information Technology,21.0,Off-Campus,2.3,20.0,4,59,2.37,6.6,2022-10-24,0,Off-Campus,106054


In [21]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
